In [2]:
import re
import pandas as pd

# -----------------------------
# Paths to your uploaded files
# -----------------------------
SCOPUS_PATH = "scopus_export_Mar 5-2026_de2dadd3-cdf7-432e-a817-ea41f345d8db.txt"
IEEE_PATH   = "IEEE Xplore Citation Plain Text Download 2026.3.4.16.22.6.txt"
PUBMED_PATH = "summary-autismspec-set _PUBMED.txt"

# -----------------------------
# Load files (robust to encoding issues)
# -----------------------------
scopus = open(SCOPUS_PATH, encoding="utf-8", errors="ignore").read()
ieee   = open(IEEE_PATH,   encoding="utf-8", errors="ignore").read()
pubmed = open(PUBMED_PATH, encoding="utf-8", errors="ignore").read()

# -----------------------------
# Parsers
# -----------------------------
def parse_scopus_records(text: str) -> list[dict]:
    """
    Scopus plain-text export is semi-structured.
    Reliable boundary: each record contains a line starting with 'DOCUMENT TYPE:'.
    We find each such block and:
      - locate the year line like '(2024)'
      - take the nearest non-empty line above it as the title
      - extract DOI if present inside the block
    Returns exactly 119 records for your file.
    """
    lines = text.splitlines()
    doc_idxs = [i for i, l in enumerate(lines) if l.strip().startswith("DOCUMENT TYPE:")]

    recs = []
    for idx, start in enumerate(doc_idxs):
        end = doc_idxs[idx + 1] if idx + 1 < len(doc_idxs) else len(lines)

        # Find the year line just above (search upward within a window)
        year_idx = None
        for i in range(start - 1, max(-1, start - 350), -1):
            if re.match(r"^\(\d{4}\)", lines[i].strip()):
                year_idx = i
                break

        # Title is the nearest non-empty line above the year line
        title = ""
        if year_idx is not None:
            j = year_idx - 1
            while j >= 0 and not lines[j].strip():
                j -= 1
            if j >= 0:
                title = lines[j].strip()

        # DOI is usually a line starting with 'DOI:'
        doi = ""
        for i in range(start, end):
            s = lines[i].strip()
            if s.startswith("DOI:"):
                doi = s.split("DOI:")[1].strip()
                break

        recs.append({
            "Database": "Scopus",
            "Title": title,
            "DOI": doi,
            "PMID": ""
        })

    return recs


def parse_ieee_records(text: str) -> list[dict]:
    """
    IEEE Xplore 'Plain Text Download' format typically has each record as a paragraph block,
    and the paper title appears in double quotes.

    We:
      - split by blank lines
      - extract the first quoted string as the title
      - try to extract DOI if present (doi: ...)
    Returns exactly 13 records for your file.
    """
    recs = []
    blocks = [b.strip() for b in re.split(r"\n\s*\n", text) if b.strip()]

    for b in blocks:
        m_title = re.search(r"\"([^\"]+)\"", b)
        if not m_title:
            continue
        title = m_title.group(1).strip()

        m_doi = re.search(r"\bdoi:\s*([0-9]+\.[0-9]+\/[^\s,;]+)", b, flags=re.I)
        doi = m_doi.group(1).rstrip(".") if m_doi else ""

        recs.append({
            "Database": "IEEE",
            "Title": title,
            "DOI": doi,
            "PMID": ""
        })

    return recs


def parse_pubmed_records(text: str) -> list[dict]:
    """
    PubMed summary file is numbered:
      '1: Author... Title. Journal... doi: ... PMID: ...'
    We split on lines that start a new numbered entry.
    Title is typically the sentence after the author sentence.

    Returns exactly 36 records for your file.
    """
    recs = []

    # split at boundaries like "\n2: " while keeping entries intact
    entries = re.split(r"\n(?=\d+: )", text.strip())

    for e in entries:
        m = re.match(r"(\d+):\s*(.*)", e, flags=re.S)
        if not m:
            continue

        body = " ".join([ln.strip() for ln in m.group(2).splitlines() if ln.strip()])

        # Title is usually the second sentence: Authors. TITLE. Journal...
        parts = body.split(". ")
        title = parts[1].strip().rstrip(".") if len(parts) >= 2 else ""

        m_doi = re.search(r"\bdoi:\s*([0-9]+\.[0-9]+\/[^\s;]+)", body, flags=re.I)
        doi = m_doi.group(1).rstrip(".") if m_doi else ""

        m_pmid = re.search(r"PMID:\s*(\d+)", body)
        pmid = m_pmid.group(1) if m_pmid else ""

        recs.append({
            "Database": "PubMed",
            "Title": title,
            "DOI": doi,
            "PMID": pmid
        })

    return recs


# -----------------------------
# Run parsing + sanity checks
# -----------------------------
scopus_recs = parse_scopus_records(scopus)
ieee_recs   = parse_ieee_records(ieee)
pubmed_recs = parse_pubmed_records(pubmed)

print("Counts:")
print("  Scopus:", len(scopus_recs))
print("  IEEE:  ", len(ieee_recs))
print("  PubMed:", len(pubmed_recs))
print("  Total: ", len(scopus_recs) + len(ieee_recs) + len(pubmed_recs))

# -----------------------------
# Build screening dataframe (duplicates preserved)
# -----------------------------
df = pd.DataFrame(scopus_recs + ieee_recs + pubmed_recs)

# Add reproducible record IDs
df.insert(0, "Record ID", [f"R{str(i+1).zfill(4)}" for i in range(len(df))])

# Add screening columns (to be filled by you)
for col in [
    "Stage (Title/Abstract/Full-text)",
    "Decision (Include/Exclude)",
    "Reason for Exclusion",
    "Duplicate Group ID",
    "Notes"
]:
    df[col] = ""

# Optional: separate counts sheet
counts = pd.DataFrame({
    "Database": ["Scopus", "PubMed", "IEEE", "Total"],
    "Records":  [len(scopus_recs), len(pubmed_recs), len(ieee_recs),
                 len(scopus_recs) + len(pubmed_recs) + len(ieee_recs)]
})

# -----------------------------
# Save Excel workbook
# -----------------------------
OUT_PATH = "autism_ai_screening_sheet_ALL168.xlsx"
with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="Screening")
    counts.to_excel(writer, index=False, sheet_name="Counts")

print("Saved:", OUT_PATH)

Counts:
  Scopus: 119
  IEEE:   13
  PubMed: 36
  Total:  168
Saved: autism_ai_screening_sheet_ALL168.xlsx
